In [2]:
import json
import numpy as np
from pathlib import Path
from datasets import load_dataset

# Your exact absolute paths
OUTPUT_DIR = Path("/Users/farhanwajid/Web Development/SIH/SatQuery-AI---An-Interactive-Vision-Language-Assistant-for-Multimodal-Remote-Sensing-Image-Analysis-/FineTuning-VLM-LLM/model_training/data_prep/Outputs")
OUTPUT_FILE = OUTPUT_DIR / "cdvqa_train.jsonl"
IMAGE_CACHE_DIR = Path("/Users/farhanwajid/Web Development/SIH/downloaded_benchmarks/cdvqa_cache")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("📥 PIVOT: Streaming Bi-Temporal Change Detection dataset using 'blanchon/LEVIR_CDPlus'...")

try:
    # We are pivoting to the highly stable LEVIR_CDPlus repository
    dataset = load_dataset("blanchon/LEVIR_CDPlus", split="train", streaming=True)
    
    # Manually controlling the iterator to prevent server crashes
    dataset_iter = iter(dataset)
    
    samples = []
    MAX_SAMPLES = 500  # LEVIR_CDPlus has 985 pairs total, 500 is perfect for our hackathon LoRA run

    idx = 0
    while len(samples) < MAX_SAMPLES:
        try:
            item = next(dataset_iter)
        except StopIteration:
            break
        except Exception as e:
            print(f"  [⚠️ Skipping row due to stream error]: {e}")
            continue

        # Dynamically find the image keys (Authors name these differently: image1, image_A, A, etc.)
        keys = list(item.keys())
        img_a_key = next((k for k in keys if k in ['image1', 'image_A', 'A', 'pre_image']), None)
        img_b_key = next((k for k in keys if k in ['image2', 'image_B', 'B', 'post_image']), None)
        mask_key = next((k for k in keys if k in ['label', 'mask', 'change_mask']), None)

        if not img_a_key or not img_b_key or not mask_key:
            continue

        img_a = item[img_a_key]
        img_b = item[img_b_key]
        mask = item[mask_key]

        # Save both temporal images locally
        img_a_path = IMAGE_CACHE_DIR / f"cdvqa_{idx}_t1.png"
        img_b_path = IMAGE_CACHE_DIR / f"cdvqa_{idx}_t2.png"
        
        if not img_a_path.exists():
            img_a.save(img_a_path)
        if not img_b_path.exists():
            img_b.save(img_b_path)

        # ---------------------------------------------------------
        # Generate the VQA Text based on the actual Change Mask
        # ---------------------------------------------------------
        # Convert mask to a grayscale numpy array
        mask_array = np.array(mask.convert("L"))
        
        # If there are more than 500 white pixels (changed pixels), structural changes occurred
        change_pixels = np.sum(mask_array > 128)
        
        if change_pixels > 500:
            answer = "Significant structural changes are detected between the two dates, indicating new urban construction or terrain modification."
        else:
            answer = "No significant structural changes were detected between the two observation dates. The terrain remains largely unchanged."

        # Format into Qwen2-VL Multi-Image Schema
        entry = {
            "id": f"cdvqa_{idx}",
            "images": [str(img_a_path.resolve()), str(img_b_path.resolve())],
            "conversations": [
                {
                    "role": "user",
                    "content": f"<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|>\nCompare these two satellite images taken at different times. Describe any structural changes that occurred."
                },
                {
                    "role": "assistant",
                    "content": answer
                }
            ]
        }
        samples.append(entry)
        
        if (idx + 1) % 100 == 0:
            print(f"  • Processed {idx + 1} / {MAX_SAMPLES} samples...")
            
        idx += 1

    # Save to JSONL
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for entry in samples:
            f.write(json.dumps(entry) + "\n")

    print(f"\n✅ Prepared {len(samples)} Bi-Temporal Change pairs at: {OUTPUT_FILE}")

except Exception as e:
    print(f"⚠️ Critical Error occurred: {e}")

📥 PIVOT: Streaming Bi-Temporal Change Detection dataset using 'blanchon/LEVIR_CDPlus'...


  • Processed 100 / 500 samples...
  • Processed 200 / 500 samples...
  • Processed 300 / 500 samples...
  • Processed 400 / 500 samples...
  • Processed 500 / 500 samples...

✅ Prepared 500 Bi-Temporal Change pairs at: /Users/farhanwajid/Web Development/SIH/SatQuery-AI---An-Interactive-Vision-Language-Assistant-for-Multimodal-Remote-Sensing-Image-Analysis-/FineTuning-VLM-LLM/model_training/data_prep/Outputs/cdvqa_train.jsonl
